### Please make sure that you are in the correct working directory. The working directory you should be on should have all the required necessary working files, including your dependency_map.csv, epid_ids.csv, and all python scripts.

In [ ]:
# Checking Working Directory
!pwd

In [ ]:
# Changing Working Directory
!cd /mnt/share/homes/chrish47/live_birth_adjustments

---

# Maternal Non-fatal Outcomes --------------------------------------------------------

### The following code sets up the main output destination and introduces new object names into the environment.

#### IMPORTANT: Please make sure that your `dependency_map.csv` and `epi_ids.csv` files are updated.
* epid_ids.csv gets used in the save.py script

#### In the code please change the following accordingly. These are important:  
#### 1. error_path
#### 2. output_path  
#### 3. root  
#### 4. release_id and yearvals [GBD round and year data]

In [25]:
import os
import re
import time
import getpass
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
from db_queries import get_best_model_versions
import gbd.constants as gbd

### TODO: Add a way to submit Zero_Fistula and Fistula jobs simultaneously.
###       Currently, the job fails because the me_id for Fistula does not have 
###       a model version id returned by get_model_versions, which is used for 
###       the adjustments job name. Workaround: run Zero_Fistula first (with zero=True), 
###       then run Fistula (with zero=False).
################################################################################
# Please read the readme.txt in this repo. It explains the entire strategy.
################################################################################

# Set round id, years to execute, and boolean for fistula
release_id = 16
yearvals = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2022, 2023, 2024]  # Change as necessary
zero = True  # Set True or False for fistula

# Set up stdout and stderr
username = getpass.getuser()
root = os.path.join('/share/scratch/users/', username)
error_path = "/mnt/share/scratch/users/chrish47/errors/%x.e%j"  # Modify as necessary
output_path = "/mnt/share/scratch/users/chrish47/output/%x.o%j"  # Modify as necessary

# Create directories if they don't exist
if not os.path.exists(error_path):
    os.makedirs(error_path)
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Define the path to the nonfatal_folder
nonfatal_folder = os.path.join(root, 'nonfatal_maternal')

# Check if the nonfatal_folder exists, and if not, create it
if not os.path.exists(nonfatal_folder):
    os.makedirs(nonfatal_folder)
    print(f"'{nonfatal_folder}' has been created.")
else:
    print(f"'{nonfatal_folder}' already exists.")

# Pull in dependency map
dep_map = pd.read_csv('{}/dependency_map.csv'.format(os.getcwd()))

# Create a new DataFrame that converts all values in dep_map to ints and 
# matches each output_me with its input_mes
columns = ['input_me', 'output_me']
extended_map = pd.DataFrame(columns=columns)

for index, row in dep_map.iterrows():
    ins = [int(x) for x in str(row.input_me).split(';')]
    outs = [int(x) for x in str(row.output_mes).split(';')]
    df = pd.DataFrame(columns=columns, data=list(zip(np.repeat(ins, len(outs)), outs * len(ins))))
    extended_map = pd.concat([df, extended_map], ignore_index=True)

# Create timestamped output folders
date_regex = re.compile(r'\W')
date_unformatted = str(datetime.now())[0:13]
c_date = date_regex.sub('_', date_unformatted)

# Automatic data folder vs Manual
base_dir = os.path.join(root, 'nonfatal_maternal', f'{c_date}')
#base_dir = os.path.join(root, 'nonfatal_maternal', '2024_09_11_18')  # Manual sourcing option
print(f"The following folder and its components were created: {base_dir}")

# Create directories for each unique output_me
for output_me in extended_map['output_me'].unique():
    out_dir = os.path.join(base_dir, str(output_me))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

# Grab version ids for all input models. This assumes no new dismod source models 
# have been marked best since the start of the LBA code.
model_version_list = extended_map['input_me'].unique()
mvid_df = get_best_model_versions(entity='modelable_entity', ids=model_version_list, release_id=release_id)
mvid_df.head()

'/share/scratch/users/chrish47/nonfatal_maternal' already exists.
The following folder and its components were created: /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13


---

### Zero out non-fistula

##### The current modeling strategy for Fistula uses DisMod data for a subset of the GBD locations and sets the rest of the locations to zero. The areas for which DisMod data are used are most detailed locations in Sub-Saharan Africa (SSA); most detailed locations in South Asia (SA); and Afghanistan, Yemen, and Sudan. The zero_locs method filters Dismod data to the appropriate locations using a pre-prepared Stata file. The Stata file must be updated annually to accommodate any changes to the location hierarchy. Currently, it is the responsibility of the Fistula researcher to update the Stata file.

##### 1. Please first run 'Zero out non-fistula locations - Preparring Draws'
##### 2. Once the above jobs have been completed, proceed with running 'Zero out non-fistula locations - Saving/Uploading Draws to the Epi Database'.

In [ ]:
################################################################################
# Zero out non-fistula locations - Preparring Draws
################################################################################

if zero:
    class_name = "Zero_Fistula"
    input_me  = int(dep_map.loc[dep_map.class_name==class_name,
        'input_me'].item())
    output_me = int(dep_map.loc[dep_map.class_name==class_name,
        'output_mes'].item())

    # zero out locations
    job_list = []
    
    for year in yearvals:
        job_name = "{cn}_{y}".format(cn=class_name, y=year)
        job_list.append(job_name)
        call = ('sbatch -c 10 --mem=50G'
                ' -p long.q -C archive'
                ' -A proj_rgud'
                ' -t 12:30:00' 
                ' -o {o}'
                ' -e {e}'
                ' -J {jn}'
                ' python_shell.sh'
                ' maternal_core.py'
                ' "{arg1}" "{arg2}" "{arg3}" "{arg4}" "{arg5}" "{arg6}" '.format(
                    o=output_path, e=error_path, jn=job_name, 
                    arg1=class_name, arg2=base_dir, arg3=year, 
                    arg4=input_me, arg5=output_me, arg6=release_id))
        subprocess.check_output(call, shell=True)
        print(call)

In [ ]:
################################################################################
# Zero out non-fistula locations - Saving/Uploading Draws to the Epi Database
################################################################################

if zero:
    class_name = "Zero_Fistula"
    input_me  = int(dep_map.loc[dep_map.class_name==class_name,
        'input_me'].item())
    output_me = int(dep_map.loc[dep_map.class_name==class_name,
        'output_mes'].item())

    # zero out locations
    job_list = []

    #hold = ",".join(job_list)
    model_version_ids = str(mvid_df.loc[mvid_df.modelable_entity_id==input_me, 
        'model_version_id'].item())
    save_job_name = "save_{meid}".format(meid=output_me)
    out_dir = ('{bd}/{meid}'.format(bd=base_dir, meid=output_me))
    call = ('sbatch -c 25 --mem=80G'
            ' -p long.q -C archive'
            ' -A proj_rgud'
            ' -t 12:30:00'
            ' -o {o}'
            ' -e {e}'
            ' -J {jn}'
            ' python_shell.sh'
            ' save.py'
            ' {arg1} \'{arg2}\' {arg3} {arg4}'.format(o=output_path, 
                e=error_path, jn=save_job_name, arg1=output_me, 
                arg2=model_version_ids, arg3=out_dir, arg4=release_id))
    subprocess.check_output(call, shell=True)
    print(call)

---

### Maternal Causes - Adjust maternal causes for live births

##### The following section has three important parts that must be run in a orderly fashion.  
##### This is organized by section, as its easier to debug, and to run independently for testing purposes.  

##### Please run the following orderly one by one until completion:  
* 1. 'Adjust Maternal Causes for Live Births - Preparring Draws'
* 2. 'Uploading Epi Bundle data'
* 3. 'Maternal non-fatal outcomes locations - Saving/Uploading Draws to the Epi Database'

In [ ]:
################################################################################
# Maternal Causes Live Birth Adjustement - Preparring Draws
################################################################################

# Removing the Zero_Fistula class or it will be uploaded twice
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []

for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']
    
    for year in yearvals:
        job_name = 'adj_{cn}_{y}'.format(cn=class_name, y=year)
        """ this line is to facilitate testing one class at a time, 
        change as needed """
        if (class_name not in ["abcdefghijk..."]):
            adjust_job_list.append(job_name)
            call = ('sbatch -c 10 --mem=80G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J {jn}'
                    ' python_shell.sh'
                    ' maternal_core.py'
                    ' {arg1} {arg2} {arg3} "{arg4}" "{arg5}" {arg6}'.format(
                        o=output_path, e=error_path, jn=job_name, 
                        arg1=class_name, arg2=base_dir, arg3=year, 
                        arg4=input_me, arg5=output_me, arg6=release_id))
            subprocess.check_output(call, shell=True)
            print(call)

---

In [ ]:
# ############################################################################
# # Uploading Epi Bundle data
# # For these bundles only:
# # * 377 ~ Infertility due to puerperal sepsis
# # * 827 ~ Long term sequelae of eclampsia
# # * 829 ~ Long term sequelae of severe pre-eclampsia
# ############################################################################

dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []

for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    dismod_dict = {
        2624 : ('maternal_sepsis', 377),
        2627 : ('maternal_htn', 827),
        1546 : ('maternal_htn', 829)
    }
    output_mes = [int(x) for x in str(output_me).split(';')]
    epi_upload_list = [x for x in output_mes if x in list(dismod_dict.keys())]
    
    for output_me in epi_upload_list:
        bundle_tuple = dismod_dict[output_me]
        """ this line is to facilitate testing one bundle at a time, 
        change to == as needed """
        if (int(bundle_tuple[1]) > -1):
            data_dir = os.path.join(base_dir, str(output_me))
            call = ('sbatch -c 3 --mem=10G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J upload_bundle_{jn}'
                    ' python_shell.sh'
                    ' upload_maternal_epi_bundles.py'
                    ' {arg1} {arg2} "{arg3}" '.format(o=output_path,# Deleted decomp_step, no longer needed for elmo functions 
                        e=error_path, jn=bundle_tuple[1], arg1=data_dir, 
                        arg2=bundle_tuple[0], arg3=bundle_tuple[1]))
            subprocess.check_output(call, shell=True)
            print(call)

In [ ]:
############################################################################
# Maternal non-fatal outcomes locations - Saving/Uploading Draws to the Epi Database
############################################################################

dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []

for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    output_mes = [int(x) for x in str(output_me).split(';')]

    save_mes = [x for x in output_mes if x not in list(dismod_dict.keys())]
    for me in save_mes:
        data_dir = os.path.join(base_dir, str(me))

        if me not in [-1]:
            model_ids_str_for_save = ''
            df = extended_map.loc[extended_map.output_me==me,:]
            for index,row in df.iterrows():
                model_ids_str_for_save += ' meid {}, mvid {};'.format(
                    row.input_me, 
                    mvid_df.loc[mvid_df.modelable_entity_id==row.input_me,
                    'model_version_id'].item())
            model_ids_str_for_save = model_ids_str_for_save[:-1] + "."
            call = ('sbatch -c 25 --mem=80G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J save_{jn}'
                    ' python_shell.sh'
                    ' save.py'
                    ' {arg1} \'{arg2}\' {arg3} "{arg4}" '.format(o=output_path, 
                        e=error_path, jn=me, arg1=me, 
                        arg2=model_ids_str_for_save, 
                        arg3=data_dir, arg4=release_id))
            subprocess.check_output(call, shell=True)
            print(call)

    sleeptime = 5
    print("Sleeping for {} seconds".format(sleeptime))
    time.sleep(sleeptime)

# GDM and PPCM --------------------------------------------------------

### The following code sets up the main output destination and introduces new object names into the environment.

#### IMPORTANT: The following code is documented for GDM and PPCM. But it can work for other causes if needed to run independently.

#### IMPORTANT: Please make sure that your `dependency_map_{cause}.csv` and `epi_ids_{cause}.csv` files are updated.
* epid_ids.csv gets used in the save.py script  

#### In the code please change the following accordingly. These are important:  
##### 1. error_path
##### 2. output_path  
##### 3. root  
##### 4. release_id and yearvals [GBD round and year data]
##### 5. Please change your dependecy map `dependency_map_{cause}.csv` accordingly.

In [30]:
import os
import re
import time
import getpass
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
from db_queries import get_best_model_versions
import gbd.constants as gbd

### TODO: Add a way to submit Zero_Fistula and Fistula jobs simultaneously.
###       Currently, the job fails because the me_id for Fistula does not have 
###       a model version id returned by get_model_versions, which is used for 
###       the adjustments job name. Workaround: run Zero_Fistula first (with zero=True), 
###       then run Fistula (with zero=False).
################################################################################
# Please read the readme.txt in this repo. It explains the entire strategy.
################################################################################

# Set round id, years to execute, and boolean for fistula
release_id = 33
yearvals = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2022, 2023, 2024]  # Change as necessary
zero = True  # Set True or False for fistula

# Set up stdout and stderr
username = getpass.getuser()
root = os.path.join('/share/scratch/users/', username)
error_path = "/mnt/share/scratch/users/chrish47/errors/%x.e%j"  # Modify as necessary
output_path = "/mnt/share/scratch/users/chrish47/output/%x.o%j"  # Modify as necessary

# Create directories if they don't exist
if not os.path.exists(error_path):
    os.makedirs(error_path)
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Define the path to the nonfatal_folder
nonfatal_folder = os.path.join(root, 'nonfatal_maternal')

# Check if the nonfatal_folder exists, and if not, create it
if not os.path.exists(nonfatal_folder):
    os.makedirs(nonfatal_folder)
    print(f"'{nonfatal_folder}' has been created.")
else:
    print(f"'{nonfatal_folder}' already exists.")

# Pull in dependency map
dep_map = pd.read_csv('{}/dependency_map_gdm.csv'.format(os.getcwd())) ###CHANGE ACCORDINGLY

# Create a new DataFrame that converts all values in dep_map to ints and 
# matches each output_me with its input_mes
columns = ['input_me', 'output_me']
extended_map = pd.DataFrame(columns=columns)

for index, row in dep_map.iterrows():
    ins = [int(x) for x in str(row.input_me).split(';')]
    outs = [int(x) for x in str(row.output_mes).split(';')]
    df = pd.DataFrame(columns=columns, data=list(zip(np.repeat(ins, len(outs)), outs * len(ins))))
    extended_map = pd.concat([df, extended_map], ignore_index=True)

# Create timestamped output folders
date_regex = re.compile(r'\W')
date_unformatted = str(datetime.now())[0:13]
c_date = date_regex.sub('_', date_unformatted)

# Automatic data folder vs Manual
base_dir = os.path.join(root, 'nonfatal_maternal', f'{c_date}')
#base_dir = os.path.join(root, 'nonfatal_maternal', '2025_04_11_20')  # Manual sourcing option
print(f"The following folder and its components were created: {base_dir}")

# Create directories for each unique output_me
for output_me in extended_map['output_me'].unique():
    out_dir = os.path.join(base_dir, str(output_me))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

# Grab version ids for all input models. This assumes no new dismod source models 
# have been marked best since the start of the LBA code.
model_version_list = extended_map['input_me'].unique()
mvid_df = get_best_model_versions(entity='modelable_entity', ids=model_version_list, release_id=release_id) 
mvid_df.head()
# Change model_version_Id as necessary OR Delete it. I had to input this for GDM. 
# MAKE SURE YOU CHECK WITH THE MODELER ABOUT THE MODEL VERSION ID. The BESTED model will get used.

'/share/scratch/users/chrish47/nonfatal_maternal' already exists.
The following folder and its components were created: /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13


,cause_id,cause_name,covariate_id,covariate_name,rei_id,rei_name,modelable_entity_id,modelable_entity_name,model_type_id,model_type,model_version_id,release_id,best_start,date_inserted,description
0,1118,Gestational diabetes,None,None,None,None,27714,Gestational diabetes mellitus in pregnancy (ev...,3,dismod,926214,33,2025-04-25 18:08:18,2025-04-25 07:47:40,"4/18 denom-adj, diagnosis-xwalk other_notwalke..."


#### Please change the following accordingly:
##### 1. `maternal_core.py` or `maternal_core_ppcm.py`
* `maternal_core.py` is for GDM and any other cause you might like to run independently.
* `maternal_core_ppcm.py` is only for PPCM.

In [32]:
################################################################################
# Maternal Causes Live Birth Adjustement - Preparring Draws
################################################################################

adjust_job_list = []
for index, row in dep_map.iterrows():
    class_name = row['class_name']
    #print(class_name)
    input_me = row['input_me']
    #print(input_me)
    output_me = row['output_mes']
    #print(output_me)

    for year in yearvals:
        job_name = 'adj_{cn}_{y}'.format(cn=class_name, y=year)
        """ this line is to facilitate testing one class at a time, 
        change as needed """
        if (class_name not in ["abcdefghijk..."]):
            adjust_job_list.append(job_name)
            call = ('sbatch -c 10 --mem=50G'
                    ' -p all.q -C archive' #long, all
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J {jn}'
                    ' python_shell.sh'
                    ' maternal_core.py' #### Change accordingly
                    ' {arg1} {arg2} {arg3} "{arg4}" "{arg5}" {arg6}'.format(
                        o=output_path, e=error_path, jn=job_name, 
                        arg1=class_name, arg2=base_dir, arg3=year, 
                        arg4=input_me, arg5=output_me, arg6=release_id))
            subprocess.check_output(call, shell=True)
            print(call)

sbatch -c 10 --mem=50G -p all.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_GDM_1990 python_shell.sh maternal_core.py GDM /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13 1990 "27714" "27715" 33
sbatch -c 10 --mem=50G -p all.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_GDM_1995 python_shell.sh maternal_core.py GDM /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13 1995 "27714" "27715" 33
sbatch -c 10 --mem=50G -p all.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_GDM_2000 python_shell.sh maternal_core.py GDM /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13 2000 "27714" "27715" 33
sbatch -c 10 --mem=50G -p all.q -C archive -A proj_rgud -t 12:30:00 -o /m

#### Please change the following accordingly:
##### 1. `save_gdm.py` or `save_ppcm_ppcm.py`
* `save_gdm.py` is for GDM
* `save_ppcm.py` is only for PPCM.

##### IMPORTANT: If you are running another cause besides GDM or PPCM, you can simply modify one of the above save_{cause}.py scripts to run with the correct `epi_ids_{cause}.csv` file.
* This is on the to do list for future updates.

In [37]:
############################################################################
# Save results in parallel for everything except dismod_dict keys
############################################################################

for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    # Process output_mes directly without using dismod_dict
    output_mes = [int(x) for x in str(output_me).split(';')]
    
    # Directly create save_mes without any reference to dismod_dict
    save_mes = output_mes

    for me in save_mes:
        data_dir = os.path.join(base_dir, str(me))
        """ this line is to facilitate testing one bundle at a time, 
        change as needed """
        print(data_dir)
        
        if me not in [-1]:
            model_ids_str_for_save = ''
            df = extended_map.loc[extended_map.output_me == me, :]
            
            for index, row in df.iterrows():
                model_ids_str_for_save += ' meid {}, mvid {};'.format(
                    row.input_me, 
                    mvid_df.loc[mvid_df.modelable_entity_id == row.input_me,
                                'model_version_id'].item())
            
            model_ids_str_for_save = model_ids_str_for_save[:-1] + "."
            
            call = ('sbatch -c 25 --mem=80G'
                    ' -p all.q -C archive' #all#long
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J save_{jn}'
                    ' python_shell.sh'
                    ' save_gdm.py' #### Change accordingly
                    ' {arg1} \'{arg2}\' {arg3} "{arg4}" '.format(o=output_path, 
                        e=error_path, jn=me, arg1=me, 
                        arg2=model_ids_str_for_save, 
                        arg3=data_dir, arg4=release_id))
            subprocess.check_output(call, shell=True)
            print(call)

/share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13/27715
sbatch -c 25 --mem=80G -p all.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J save_27715 python_shell.sh save_gdm.py 27715 ' meid 27714, mvid 926214.' /share/scratch/users/chrish47/nonfatal_maternal/2025_04_25_13/27715 "33" 
